PePy

Author: Julia K. Varga <jvarga92@gmail.com>  
License: BSD 3 clause  
Code Repository: https://github.com/gezmi/pepy

In [1]:
import pandas as pd
pd.set_option('display.width', 600)
pd.set_option('display.max_columns', 10)

# Full Analysis Pipeline

PePy can run the complete analysis — interface calculation, confidence loading, and metrics — in a single call with `calculate_all_metrics()`.

This tutorial shows the one-call workflow and how to process multiple structures in batch.

In [2]:
from pepy import ProteinComplex

## One-Call Workflow

`calculate_all_metrics()` performs:
1. `calculate_interface()` — if not already done
2. `load_confidence_data()` — if not already loaded
3. `calculate_confidence_metrics()` — if confidence data is available

All parameters are passed through to the underlying methods:

In [3]:
cx = ProteinComplex.from_file(
    '../../pepy/tests/data/1ycr_af2_55d19_unrelaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000.pdb'
)
cx.identify_chains()

results = cx.calculate_all_metrics(
    cb_cutoff=8.0,
    all_atom_cutoff=4.0,
    confidence_threshold=70.0,
    require_confidence=False,  # don't fail if no JSON found
)

Binder chain(s): B, receptor chain(s): A


### What's in the result?

In [4]:
print('Top-level keys:', list(results.keys()))

Top-level keys: ['interface_residues', 'confidence_loaded', 'metrics', 'summary']


In [5]:
# Interface residues
print('Binder residues:', results['interface_residues']['binder'])
print('Receptor residues:', results['interface_residues']['receptor'])

Binder residues: [2, 3, 4, 5, 8, 9, 11, 12, 13, 14]
Receptor residues: [35, 38, 42, 45, 51, 54, 56, 57, 59, 77, 80, 83, 84]


In [6]:
# Confidence metrics
results['metrics']

{'status': 'calculated',
 'avg_plddt_interface': 91.68,
 'max_plddt_interface': 98.19,
 'interface_pae': 1.97,
 'min_interface_pae': 0.95,
 'iptm': 0.85,
 'ptm': 0.78,
 'combined_confidence': 0.84,
 'has_interface_metrics': True}

In [7]:
# Full summary (structure + interface + confidence)
results['summary']

{'structure_info': {'status': 'ready',
  'binder_chains': ['B'],
  'receptor_chains': ['A'],
  'binder_length': 15,
  'receptor_length': 109,
  'total_chains': 2,
  'unassigned_chains': []},
 'interface_info': {'status': 'calculated',
  'binder_interface_residues': 10,
  'receptor_interface_residues': 13,
  'binder_interface_atoms': 90,
  'receptor_interface_atoms': 113,
  'binder_residue_list': [2, 3, 4, 5, 8, 9, 11, 12, 13, 14],
  'receptor_residue_list': [35,
   38,
   42,
   45,
   51,
   54,
   56,
   57,
   59,
   77,
   80,
   83,
   84]},
 'confidence_info': {'status': 'confidence_loaded',
  'has_pae_matrix': True,
  'has_iptm': True,
  'has_ptm': True,
  'iptm': 0.85,
  'ptm': 0.78,
  'combined_confidence': 0.84},
 'metrics': {'status': 'calculated',
  'avg_plddt_interface': 91.68,
  'max_plddt_interface': 98.19,
  'interface_pae': 1.97,
  'min_interface_pae': 0.95,
  'iptm': 0.85,
  'ptm': 0.78,
  'combined_confidence': 0.84,
  'has_interface_metrics': True}}

## Batch Processing

A common use case is processing a directory of prediction outputs and collecting results into a table.

Here's a pattern using the test data as an example:

In [8]:
import os
import glob

data_dir = '../../pepy/tests/data'
structure_files = (
    glob.glob(os.path.join(data_dir, '*.pdb')) +
    glob.glob(os.path.join(data_dir, '*.cif'))
)

# Skip the plain PDB (no second chain type info)
structure_files = [f for f in structure_files if '1YCR' not in os.path.basename(f)]

print(f'Found {len(structure_files)} structure files:')
for f in structure_files:
    print(f'  {os.path.basename(f)}')

Found 3 structure files:
  1ycr_af2_55d19_unrelaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000.pdb
  pred.model_idx_0.pdb
  fold_1ycr_af3_model_0.cif


In [9]:
rows = []

for filepath in structure_files:
    basename = os.path.basename(filepath)
    try:
        cx = ProteinComplex.from_file(filepath)
        cx.identify_chains()
        results = cx.calculate_all_metrics(require_confidence=False)

        metrics = results['metrics']
        row = {
            'file': basename,
            'binder': ','.join(cx.binder_chains),
            'receptor': ','.join(cx.receptor_chains),
            'n_binder_res': len(results.get('interface_residues', {}).get('binder', [])),
            'n_receptor_res': len(results.get('interface_residues', {}).get('receptor', [])),
        }

        if metrics['status'] == 'calculated':
            row.update({
                'avg_plddt': metrics['avg_plddt_interface'],
                'interface_pae': metrics['interface_pae'],
                'iptm': metrics['iptm'],
                'confidence': metrics['combined_confidence'],
            })

        rows.append(row)

    except Exception as e:
        rows.append({'file': basename, 'error': str(e)})

df = pd.DataFrame(rows)
df

Binder chain(s): B, receptor chain(s): A


Binder chain(s): B, receptor chain(s): A


Binder chain(s): B, receptor chain(s): A


,file,binder,receptor,n_binder_res,n_receptor_res,avg_plddt,interface_pae,iptm,confidence
0,1ycr_af2_55d19_unrelaxed_rank_001_alphafold2_m...,B,A,10,13,91.68,1.97,0.850000,0.84
1,pred.model_idx_0.pdb,B,A,9,13,92.37,30.00,0.825989,NaN
2,fold_1ycr_af3_model_0.cif,B,A,10,14,83.24,2.77,0.760000,0.76


## When to Use `calculate_all_metrics()` vs. Separate Steps

| Approach | Use when |
|----------|----------|
| `calculate_all_metrics()` | Quick analysis, batch processing, you want everything at once |
| Separate steps | You need intermediate results (e.g. interface atoms), custom logic between steps, or only need some metrics |

The separate steps are:

```python
# Step 1: Interface (no confidence needed)
cx.calculate_interface()
cx.get_interface_atoms('ca')  # intermediate access

# Step 2: Confidence (optional)
cx.load_confidence_data()

# Step 3: Metrics (needs both)
cx.calculate_confidence_metrics()
```

See the other tutorials for details on each step.

## Exporting Results

Since the batch result is a DataFrame, you can export it directly:

In [10]:
# df.to_csv('interface_results.csv', index=False)
# df.to_excel('interface_results.xlsx', index=False)
print('Ready to export — uncomment the lines above to save.')

Ready to export — uncomment the lines above to save.
